In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import coocurence

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False)
X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config, 
    X_train, 
    X_val, 
    X_val_seq, 
    ordinal_encoder, 
    return_only_val=False , 
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 15min 10s, sys: 2min 51s, total: 18min 1s
Wall time: 3min 39s


In [5]:
X_train.head()

user,aid
i64,list[i64]
4351104,"[1333093, 634707, ... 1322740]"
5043392,"[1430421, 1430421, 1660871]"
11121600,"[629182, 629182]"
2122752,"[732558, 1624783, ... 1624783]"
7538624,"[1719462, 352409, ... 409236]"


In [6]:
%%time

list_inters = list(X_train.to_pandas()[config.col_aid])

CPU times: user 3.09 s, sys: 513 ms, total: 3.6 s
Wall time: 3.56 s


In [7]:
del X_train
gc.collect()

93

In [8]:
%%time

top_triples = coocurence.train_triplet_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=10)

extracting max_candidates: 100%|███| 1898739/1898739 [00:55<00:00, 34054.83it/s]

CPU times: user 15min 8s, sys: 1min 59s, total: 17min 7s
Wall time: 17min 4s


In [11]:
%%time

rec_cooc = coocurence.CoocurenceTripletRecommender(
    top_triples=top_triples, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)

data_to_predict = X_val_seq.to_pandas()[config.col_train_seq].tolist()
df_preds = rec_cooc.recommend(data_to_predict)

df_val_users = pl.from_pandas(X_val_seq.to_pandas()[[config.col_user]].reset_index())
df_val_preds = df_val_users.join(df_preds, on='index', how='inner')
df_val_preds = df_val_preds.with_columns(
    [
        pl.col('cooc_triplet_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_triplet_rank_0'),
    ]
)

df_val_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_triplets_df_val_preds_{config.num_recs}.parquet')

predicting: 100%|███████████████████| 466825/466825 [00:01<00:00, 260238.27it/s]


CPU times: user 11 s, sys: 2.13 s, total: 13.2 s
Wall time: 11.3 s


In [12]:
del data_to_predict, df_preds, df_val_users, df_val_preds, X_val_seq, X_val
gc.collect()

228

In [13]:
del top_triples
gc.collect()

204

# TEST

In [14]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

CPU times: user 7min 46s, sys: 13min 8s, total: 20min 54s
Wall time: 2min 13s


0

In [ ]:
%%time

top_triples = coocurence.train_triplet_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=10)

filtering rare keys:  61%|███▋  | 55127953/89938518 [02:31<03:08, 184933.25it/s]

In [ ]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()
rec_cooc = coocurence.CoocurenceTripletRecommender(
    top_triples=top_triples, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)
df_preds = rec_cooc.recommend(data_to_predict)

df_test_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].reset_index())
df_test_preds = df_test_users.join(df_preds, on='index', how='inner')
df_test_preds = df_test_preds.with_columns(
    [
        pl.col('cooc_triplet_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_triplet_rank_0'),
    ]
)

df_test_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_triplets_df_test_preds_{config.num_recs}.parquet')

In [ ]:
df_test_preds